# Phase 3 EDA Charts (v2) — Spearman heatmap regen

**Group 01-02 | DS261 Spring 2026**

This is a **focused** companion to `Phase3_EDA_Charts` that **only** regenerates the
top-20 feature correlation heatmap. The original v1 chart used **Pearson** correlation,
which is inappropriate when one of the variables is a binary outcome (`DEP_DEL15`):
Pearson assumes linear relationships between continuous variables and is not the right
association measure for a binary target. We replace Pearson with **Spearman rank
correlation**, which (a) handles binary variables cleanly via ties in ranks,
(b) does not require linearity, and (c) keeps a single, business-readable color scale
across the full matrix.

**Output**: `dbfs:/FileStore/group_01_02/phase3_charts/feature_correlation_top20_spearman.png`

The original Pearson PNG is left in place for audit / diff.

In [0]:
import json
import os

import pyspark.sql.functions as F
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

BASE_GROUP    = "dbfs:/student-groups/Group_01_02"
TRAIN_PATH    = f"{BASE_GROUP}/df_train_phase3.parquet"
NUM_COLS_PATH = "/dbfs/student-groups/Group_01_02/phase3_final_num_cols.json"
CHART_DIR     = "/dbfs/FileStore/group_01_02/phase3_charts"
OUT_PNG       = f"{CHART_DIR}/feature_correlation_top20_spearman.png"

os.makedirs(CHART_DIR, exist_ok=True)

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 14,
    "axes.labelsize": 12,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
})

with open(NUM_COLS_PATH) as f:
    numerical_cols = json.load(f)
print(f"Manifest features: {len(numerical_cols)}")

df_train = spark.read.parquet(TRAIN_PATH)
print(f"Training rows: {df_train.count():,}")

## Sample, rank, and compute Spearman correlations

We sample **1%** of the training split for tractability (same sampling fraction used in
the v1 Pearson chart), then compute Spearman correlations using `pandas.DataFrame.corr(method="spearman")`.

In [0]:
# Sample 1% of training (matches v1 sampling)
sample_pdf = (
    df_train.select(["DEP_DEL15"] + numerical_cols)
            .sample(0.01, seed=42)
            .toPandas()
)
print(f"Sample rows: {len(sample_pdf):,}")

# Spearman correlations of every numeric feature with the binary target
target_corrs = sample_pdf.corr(method="spearman")["DEP_DEL15"].drop("DEP_DEL15")
top20 = target_corrs.abs().nlargest(20).index.tolist()

heatmap_cols = ["DEP_DEL15"] + top20
corr_matrix  = sample_pdf[heatmap_cols].corr(method="spearman")
print("Top-20 features by |Spearman| with DEP_DEL15:")
for f_, r_ in target_corrs.loc[top20].items():
    print(f"  {f_:35s}  rho = {r_:+.3f}")

## Render heatmap and save to DBFS

In [0]:
fig, ax = plt.subplots(figsize=(12, 10))
im = ax.imshow(corr_matrix.values, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")

ax.set_xticks(range(len(heatmap_cols)))
ax.set_yticks(range(len(heatmap_cols)))
short_names = [n[:25] for n in heatmap_cols]
ax.set_xticklabels(short_names, rotation=45, ha="right", fontsize=8)
ax.set_yticklabels(short_names, fontsize=8)

for i in range(len(heatmap_cols)):
    for j in range(len(heatmap_cols)):
        val = corr_matrix.values[i, j]
        color = "white" if abs(val) > 0.5 else "black"
        ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=6, color=color)

cbar = fig.colorbar(im, ax=ax, shrink=0.8)
cbar.set_label("Spearman Rank Correlation", fontsize=11)
ax.set_title(
    "Top 20 Features by |Spearman Rank Correlation| with DEP_DEL15\n"
    "(rank-based; valid for the binary target)",
    fontsize=14,
)
fig.tight_layout()
fig.savefig(OUT_PNG, dpi=150, bbox_inches="tight")
plt.close(fig)
print(f"Saved: {OUT_PNG}")

## Quick EDA: tail_prev_delay effect

New in the Phase 3 manifest. This cell computes the delay rate split by
`tail_prev_delay = 0` vs `tail_prev_delay = 1` from the training parquet so the report
can quote the magnitude of the same-aircraft turnaround cascade.

In [0]:
tail_stats = (
    df_train.groupBy("tail_prev_delay")
            .agg(F.mean("DEP_DEL15").alias("delay_rate"),
                 F.count("*").alias("n"))
            .orderBy("tail_prev_delay")
            .toPandas()
)
print(tail_stats.to_string(index=False))

if {0.0, 1.0}.issubset(set(tail_stats["tail_prev_delay"].astype(float))):
    r0 = float(tail_stats.loc[tail_stats["tail_prev_delay"] == 0.0, "delay_rate"].iloc[0])
    r1 = float(tail_stats.loc[tail_stats["tail_prev_delay"] == 1.0, "delay_rate"].iloc[0])
    print(f"\ndelay_rate(prev on-time) = {r0:.4f}")
    print(f"delay_rate(prev delayed) = {r1:.4f}")
    print(f"absolute lift            = {(r1 - r0):+.4f}")
    print(f"relative lift            = {(r1 / r0 - 1.0) * 100:+.1f}%")